# Single-task conditioning endpoint overfitting on Modal

This Phase 2 experiment trains one fresh full FLUX transformer for 2,048 optimizer steps on the eight fixed conditioning variants. NVARC and the FLUX VAE remain frozen. Each eight-update cycle contains one shared-noise endpoint update at t=1 and seven normally stratified updates; optimizer, batch, seed, and target set otherwise follow Phase 1.

Technical completion means that the final report and the single step-2,048 transformer checkpoint were uploaded and verified. It does not imply that the exact-grid learning criterion passed.

## 1. Immutable runtime configuration

Before running, set DIFFUSION_ARC_REVISION to the exact pushed 40-character commit containing the endpoint-overfit protocol. Branch names and missing revisions are rejected. Automatic shutdown is enabled by default; set DIFFUSION_ARC_AUTO_SHUTDOWN=0 only when the container must remain available for debugging.

In [ ]:
from __future__ import annotations

import base64
from collections import deque
from datetime import datetime, timezone
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = os.environ.get('DIFFUSION_ARC_REVISION', '').strip()
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'
PROTOCOL = 'endpoint-overfit'
RESULT_FAMILY = 'single-task-conditioning-endpoint-overfitting'
EXPECTED_STAGE = 'eight-condition-endpoint-overfit'
EXPECTED_FINAL_STEP = 2048
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-endpoint-overfit-source-{RUN_ID}'
REPORT_PATH = VOLUME_ROOT / 'results' / RESULT_FAMILY / RUN_ID / 'report.json'

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}

def require_exact_commit(revision: str) -> None:
    if len(revision) != 40 or any(
        character not in '0123456789abcdefABCDEF' for character in revision
    ):
        raise ValueError(
            'DIFFUSION_ARC_REVISION must be an exact pushed 40-character commit'
        )

def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def should_stream_runner_line(line: str) -> bool:
    normalized = line.strip()
    if not normalized:
        return False
    lowered = normalized.lower()
    return (
        normalized.startswith('[')
        or normalized.startswith(f'{EXPECTED_STAGE}:')
        or f'{EXPECTED_STAGE} step ' in normalized
        or normalized.startswith('FINAL_REPORT=')
        or any(
            marker in lowered
            for marker in ('traceback', 'error', 'failed', 'failure', 'non-finite')
        )
    )

def final_comparison_sheet(report: dict) -> Path | None:
    evaluations = report['stages'][EXPECTED_STAGE].get('evaluations', [])
    if not evaluations:
        return None
    generation = evaluations[-1].get('generation', {})
    for seed_payload in generation.get('by_seed', []):
        for key in (
            'ground_truth_prediction_sheet',
            'gt_vs_pred_comparison_sheet',
            'comparison_sheet',
            'contact_sheet',
        ):
            value = seed_payload.get(key)
            if not value:
                continue
            candidate = Path(value)
            if not candidate.is_absolute():
                candidate = REPORT_PATH.parent / candidate
            if candidate.is_file():
                return candidate
    return None

def schedule_modal_shutdown() -> None:
    if not AUTO_SHUTDOWN:
        print('Automatic shutdown disabled by DIFFUSION_ARC_AUTO_SHUTDOWN=0.')
        return
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        print('WARNING: MODAL_TASK_ID is unavailable; stop the container manually.')
        return
    shutdown_code = (
        'import subprocess, sys, time; time.sleep(15); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print('Modal shutdown scheduled in 15 seconds.')

print('Run ID:', RUN_ID)
print('Protocol:', PROTOCOL)
print('Requested source revision:', SOURCE_REVISION or '<missing>')
print('Final report:', REPORT_PATH)


## 2. Fetch, verify, train, summarize, and shut down

This single protected cell fetches a detached source commit, installs the pinned runtime only when required, runs the complete unit-test suite, launches the endpoint-overfit protocol, verifies the final report and sole checkpoint, previews the final comparison sheet when available, and schedules shutdown in a finally block after either success or failure. Per-update progress bars are suppressed from notebook output; only milestones and failures are streamed.

In [ ]:
REPORT = None
run_succeeded = False
runner_tail: deque[str] = deque(maxlen=80)

try:
    require_exact_commit(SOURCE_REVISION)
    if not VOLUME_ROOT.is_dir():
        raise FileNotFoundError(f'Modal Volume mount is missing: {VOLUME_ROOT}')
    if REPORT_PATH.exists():
        raise FileExistsError(REPORT_PATH)

    gpu_summary = run_command(
        'query GPU',
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    )
    print('GPU:', gpu_summary)

    SOURCE_DIR.mkdir(parents=True)
    run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
    run_command(
        'add origin',
        ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
        cwd=SOURCE_DIR,
    )

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment = os.environ.copy()
    git_environment.pop('GITHUB_TOKEN', None)
    git_environment.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    try:
        run_command(
            'fetch source',
            ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
            cwd=SOURCE_DIR,
            env=git_environment,
        )
        run_command(
            'checkout source',
            ['git', 'checkout', '--detach', 'FETCH_HEAD'],
            cwd=SOURCE_DIR,
        )
    finally:
        git_environment.clear()
        del credential

    resolved_source_commit = run_command(
        'resolve source',
        ['git', 'rev-parse', 'HEAD'],
        cwd=SOURCE_DIR,
    )
    if resolved_source_commit.lower() != SOURCE_REVISION.lower():
        raise RuntimeError('Fetched source commit differs from requested revision')

    run_command(
        'remove incompatible torchvision',
        [sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchvision'],
    )
    requirements = [
        f'{name}=={version}'
        for name, version in PINNED_PACKAGES.items()
        if installed_version(name) != version
    ]
    if requirements:
        run_command(
            'install pinned dependencies',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--no-input',
                '--no-cache-dir',
                *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--no-input',
            '--no-deps',
            '--no-build-isolation',
            '-e',
            str(SOURCE_DIR),
        ],
    )

    source_path = str((SOURCE_DIR / 'src').resolve())
    test_environment = os.environ.copy()
    test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
    test_environment['PYTHONPATH'] = source_path
    test_started = time.perf_counter()
    test_process = subprocess.run(
        [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
        cwd=SOURCE_DIR,
        env=test_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if test_process.returncode != 0:
        print('\n'.join(test_process.stdout.splitlines()[-50:]))
        raise RuntimeError(f'Unit tests returned {test_process.returncode}')
    print(
        f'Resolved source {resolved_source_commit}; unit tests PASS in '
        f'{time.perf_counter() - test_started:.1f}s.'
    )

    runner = SOURCE_DIR / 'scripts' / 'experiments' / 'run_single_task_conditioning_ladder.py'
    if not runner.is_file():
        raise RuntimeError(
            'Selected source revision lacks the conditioning-ladder runner'
        )
    runner_command = [
        sys.executable,
        '-u',
        str(runner),
        '--protocol',
        PROTOCOL,
        '--volume-root',
        str(VOLUME_ROOT),
        '--volume-name',
        VOLUME_NAME,
        '--run-id',
        RUN_ID,
        '--source-commit',
        resolved_source_commit,
    ]
    runner_environment = os.environ.copy()
    runner_environment.update({
        'PYTHONDONTWRITEBYTECODE': '1',
        'PYTHONPATH': source_path,
        'HF_HUB_OFFLINE': '1',
        'TRANSFORMERS_OFFLINE': '1',
        'HF_HUB_DISABLE_TELEMETRY': '1',
        'HF_HUB_DISABLE_PROGRESS_BARS': '1',
        'TOKENIZERS_PARALLELISM': 'false',
    })
    process = subprocess.Popen(
        runner_command,
        cwd=SOURCE_DIR,
        env=runner_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for raw_line in process.stdout:
        for fragment in raw_line.replace('\r', '\n').splitlines():
            normalized = fragment.strip()
            if not normalized:
                continue
            runner_tail.append(normalized)
            if should_stream_runner_line(normalized):
                print(normalized)
    return_code = process.wait()
    if return_code != 0:
        if REPORT_PATH.is_file():
            failed_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
            print(json.dumps({
                'status': failed_report.get('status'),
                'failed_stage': failed_report.get('failed_stage'),
                'failures': failed_report.get('failures'),
            }, indent=2))
        raise RuntimeError(f'Endpoint-overfit runner exited with {return_code}')

    REPORT = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    if REPORT.get('status') != 'COMPLETE':
        raise RuntimeError(f'Experiment report is not COMPLETE: {REPORT.get("status")}')
    if set(REPORT.get('stages', {})) != {EXPECTED_STAGE}:
        raise RuntimeError('Endpoint-overfit report contains an unexpected stage set')
    stage = REPORT['stages'][EXPECTED_STAGE]
    if stage.get('status') != 'COMPLETE':
        raise RuntimeError(f'{EXPECTED_STAGE} is not COMPLETE')
    update_counts = stage.get('training', {}).get('flow_sampling', {}).get(
        'update_counts'
    )
    expected_update_counts = {'shared_endpoint': 256, 'stratified': 1792}
    if update_counts != expected_update_counts:
        raise RuntimeError(
            f'Flow-audit update counts differ: {update_counts} versus '
            f'{expected_update_counts}'
        )
    evaluations = stage.get('evaluations', [])
    if not evaluations:
        raise RuntimeError('Endpoint-overfit report contains no evaluations')
    seen_panel = evaluations[-1].get('panels', {}).get('seen', {})
    if seen_panel.get('count') != 8:
        raise RuntimeError(
            f'Final seen raw count is {seen_panel.get("count")}, expected 8'
        )
    vae_eligible = seen_panel.get('vae_eligible', {})
    if vae_eligible.get('count') != 7:
        raise RuntimeError(
            'Final seen VAE-eligible count is '
            f'{vae_eligible.get("count")}, expected 7'
        )
    checkpoint = stage.get('saved_checkpoint')
    if not isinstance(checkpoint, dict):
        raise RuntimeError('Final transformer checkpoint is missing')
    if checkpoint.get('step') != EXPECTED_FINAL_STEP:
        raise RuntimeError('Final transformer checkpoint step is not 2048')
    if not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError('Final transformer checkpoint is not remotely verified')
    durability = REPORT.get('durability', {})
    if not durability.get('remote_content_verified'):
        raise RuntimeError('Final report content was not verified after upload')

    comparison = REPORT.get('comparison', {}).get(EXPECTED_STAGE)
    summary = {
        'run_id': RUN_ID,
        'source_commit': resolved_source_commit,
        'report': str(REPORT_PATH),
        'remote_report': durability.get('remote_report_path'),
        'stage': EXPECTED_STAGE,
        'checkpoint': checkpoint.get('path'),
        'checkpoint_step': checkpoint.get('step'),
        'checkpoint_state_scope': checkpoint.get('state_scope'),
        'target_exposures': stage.get('training', {}).get('target_exposures'),
        'comparison': comparison,
    }
    print(json.dumps(summary, indent=2))

    preview_path = final_comparison_sheet(REPORT)
    if preview_path is None:
        print('Final GT-vs-Pred comparison sheet is not present.')
    else:
        try:
            from IPython.display import Image as DisplayImage, display

            print('Final GT-vs-Pred comparison sheet:', preview_path)
            display(DisplayImage(filename=str(preview_path)))
        except Exception as preview_error:
            print(f'Could not preview {preview_path}: {preview_error}')
    run_succeeded = True
finally:
    if not run_succeeded and runner_tail:
        print('Runner output tail:')
        print('\n'.join(tuple(runner_tail)[-20:]))
    schedule_modal_shutdown()


## Interpretation guardrails

- Compare the final target-free exact-grid pass@1 and pass@2 with the earlier 512-step run; COMPLETE alone is not a learning result.
- Phase 2 changes both total allocation and the scheduled endpoint updates, so an improvement cannot be attributed to either intervention alone.
- Correct-versus-swapped probes establish conditioning sensitivity, not semantic generalization.
- Teacher-forced diagnostics below t=1 contain target-latent signal and must not be interpreted as normal generation.
- The clean VAE round-trip limitation for train-id remains a separately reported measurement boundary.